<!-- dads-lab-header -->
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mdehghani86/DADS5250-GenAI/blob/main/labs/M03/M03_Lab3_Function_Calling.ipynb)

![Module 3 Lab 3 - Function Calling](https://raw.githubusercontent.com/mdehghani86/DADS5250-GenAI/main/labs/M03/assets/images/M03_Lab3_Function_Calling_banner.png)

In [ ]:
# === Shared lab setup: install dads5250 + load API key + sticky pill ===
# Installs the shared utilities (pp, pretty_print, lab_pill, model constants,
# setup_openai, setup_gemini) once per Colab runtime. The same OPENAI_API_KEY
# / GEMINI_API_KEY Colab secrets are used across every DADS 5250 lab — set
# them once in the 🔑 sidebar and they're picked up automatically.
import os
import importlib.util
if importlib.util.find_spec("dads5250") is None:
    !pip install -q "git+https://github.com/mdehghani86/DADS5250-GenAI.git#subdirectory=utils"

from dads5250 import (
    pp,
    pretty_print,
    lab_pill,
    setup_openai,
    setup_gemini,
    DEFAULT_CHAT_MODEL,   # newest reasoning model that supports temperature
    DEFAULT_MINI_MODEL,   # newest mini model that supports temperature
    DEFAULT_EMBED_MODEL,  # current embeddings default
    DEFAULT_GEMINI_MODEL, # tracks the latest stable flash
)

lab_pill('M03 Lab 1 — Function Calling Techniques')            # sticky banner so you always see which lab you're in


## API check

Confirm the API connection before we start. Your key is read from a Colab Secret, an environment variable, or a hidden prompt if neither is set.

In [ ]:
# === API check: confirm the connection and show the model(s) this lab uses ===
client = setup_openai()        # loads OPENAI_API_KEY + verifies it works

pp({
    "OpenAI":      "connected",
    "chat model":  DEFAULT_CHAT_MODEL,
    "mini model":  DEFAULT_MINI_MODEL,
}, title="API check")

# From Text to Action: Function Calling

So far the model only produced **text**. Function calling lets it **take action**: you describe functions ("tools") to the model, and it decides which to call and with what arguments. You run the function and hand the result back, so the model can answer with real data or trigger real work.

The loop: **you define tools → the model requests a call → you execute it → you return the result → the model answers.**

In [ ]:
# ==========================================================
# 1. Define a tool: a function described with a JSON schema
# ==========================================================
import json

# A "tool" is a function you describe to the model. The schema tells the model
# the function's name, what it does, and what arguments it takes.
tools = [
    {
        "type": "function",
        "function": {
            "name": "get_current_weather",
            "description": "Get the current weather for a city.",
            "parameters": {
                "type": "object",
                "properties": {
                    "location": {"type": "string", "description": "City name, e.g. Boston"},
                    "unit": {"type": "string", "enum": ["celsius", "fahrenheit"],
                             "description": "Temperature unit"},
                },
                "required": ["location"],   # the model must supply a location
            },
        },
    }
]
pp(tools[0]["function"], title="Tool the model can call")

## 2. The model decides to call the tool

We pass the tools and ask a question. Instead of answering in prose, the model replies with a **tool call**: the function name plus the arguments it extracted from your text.

In [ ]:
# ==========================================================
# 2. Send a prompt with tools; the model returns a tool call
# ==========================================================
messages = [{"role": "user", "content": "What's the weather like in Boston today?"}]

response = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=messages,
    tools=tools,
    tool_choice="auto",          # let the model decide whether to call a tool
)

msg = response.choices[0].message
tool_call = msg.tool_calls[0]                     # the call the model wants to make
args = json.loads(tool_call.function.arguments)  # arguments come back as a JSON string

pp({"function": tool_call.function.name, "arguments": args},
   title="The model wants to call")

## 3. Run the function, return the result

The model can't run code, so **you** execute the function it asked for, then send the result back with role `tool`. The model then writes a natural answer grounded in that result.

In [ ]:
# ==========================================================
# 3. Execute the function and hand the result back to the model
# ==========================================================
# The real function (mocked here; in production this would hit a weather API).
def get_current_weather(location, unit="fahrenheit"):
    demo = {"Boston": 54, "Miami": 88, "Chicago": 47}
    return {"location": location, "temperature": demo.get(location, 70),
            "unit": unit, "conditions": "partly cloudy"}

result = get_current_weather(**args)             # run exactly what the model asked for

# Add the assistant's tool call and our tool result to the conversation
messages.append(msg)
messages.append({
    "role": "tool",
    "tool_call_id": tool_call.id,                # ties the result to the specific call
    "content": json.dumps(result),
})

final = client.chat.completions.create(model=DEFAULT_MINI_MODEL, messages=messages)
pretty_print(final.choices[0].message.content, title="Final answer (grounded in the tool result)")

## 4. Many tools: the model picks the right one

Give the model several tools and it chooses which fits the request (or none). Here we add a currency converter alongside the weather tool.

In [ ]:
# ==========================================================
# 4. Multiple tools: the model routes to the right one
# ==========================================================
def convert_currency(amount, from_currency, to_currency):
    rates = {("USD", "EUR"): 0.92, ("USD", "GBP"): 0.79, ("EUR", "USD"): 1.09}
    rate = rates.get((from_currency, to_currency), 1.0)
    return {"amount": amount, "from": from_currency, "to": to_currency, "result": round(amount * rate, 2)}

tools2 = tools + [{
    "type": "function",
    "function": {
        "name": "convert_currency",
        "description": "Convert an amount from one currency to another.",
        "parameters": {
            "type": "object",
            "properties": {
                "amount": {"type": "number"},
                "from_currency": {"type": "string", "description": "3-letter code, e.g. USD"},
                "to_currency": {"type": "string", "description": "3-letter code, e.g. EUR"},
            },
            "required": ["amount", "from_currency", "to_currency"],
        },
    },
}]

for question in ["How much is 100 USD in euros?", "What's the weather in Miami?"]:
    r = client.chat.completions.create(
        model=DEFAULT_MINI_MODEL,
        messages=[{"role": "user", "content": question}],
        tools=tools2, tool_choice="auto",
    )
    call = r.choices[0].message.tool_calls[0]
    pp({"question": question, "model chose": call.function.name,
        "arguments": json.loads(call.function.arguments)}, title="Tool routing")

> **Pause and think.** The model never ran any code, it only chose a tool and filled in the arguments from your sentence. Where in your own work could "turn a sentence into a validated function call" replace brittle parsing?

**Your notes** *(double-click to edit)*

- A task I would wire up as a tool: 
- What could go wrong if the model picks wrong arguments: 

## 5. Hands-on: define your own tool

Add a tool of your own (for example `book_meeting(title, date, attendees)` or `search_products(query, max_price)`), ask a question that should trigger it, and confirm the model calls it with the right arguments.

In [ ]:
# ==========================================================
# 5. Hands-on: your own tool (fill in the ----- placeholders)
# ==========================================================
my_tool = [{
    "type": "function",
    "function": {
        "name": "-----",                 # your function name
        "description": "-----",          # what it does
        "parameters": {
            "type": "object",
            "properties": {
                "-----": {"type": "-----", "description": "-----"},
            },
            "required": ["-----"],
        },
    },
}]

question = "-----"                        # a request that should trigger your tool
r = client.chat.completions.create(
    model=DEFAULT_MINI_MODEL,
    messages=[{"role": "user", "content": question}],
    tools=my_tool, tool_choice="auto",
)
call = r.choices[0].message.tool_calls[0]
pp({"model chose": call.function.name, "arguments": json.loads(call.function.arguments)},
   title="Your tool was called")

## Wrap-up

You closed the loop from text to action: **define tools → the model requests a call → you execute → you return the result → the model answers.** Combined with JSON mode and Pydantic from Lab 2, you now control the whole pipeline: what goes in (prompt), how it comes out (JSON), whether it's valid (Pydantic), and what the model can do (functions). Next module chains these into multi-step workflows.